### Introduction to Data Ingestion

In [2]:
import os
from typing import List,Dict,Any
import pandas as pd

### Import Libraries for Text Splitters

In [3]:
from langchain_core.documents import Document
from langchain_text_splitters import(
    RecursiveCharacterTextSplitter,
    CharacterTextSplitter,
    TokenTextSplitter
)
print("Setup is done")

d:\RAG\Repo-1\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Setup is done



### Understanding Document Structure

In [4]:
from langchain_core.documents import Document
doc= Document(
    page_content="This is the main text content that will be embedded and search",
    metadata={
        "source":"example.txt",
        "page":1,
        "author":"klinton",
        "date_created":"20-06-2026",
        "custom_field":"any_value"
    }
)
print("Document Structure")

print(f"Content:{doc.page_content}")
print(f"MetaData:{doc.metadata}")


Document Structure
Content:This is the main text content that will be embedded and search
MetaData:{'source': 'example.txt', 'page': 1, 'author': 'klinton', 'date_created': '20-06-2026', 'custom_field': 'any_value'}


### Text Files(.txt)-The Simplest Usecase

In [5]:
import os
os.makedirs("data/text_files",exist_ok=True)

sample_texts={
    "data/text_files/python_intro.txt": """For ssomeone with your background in Java, Spring Boot, and Genesys Cloud, Python is particularly useful for automation scripts, Terraform code generation, AWS integrations, API development, and AI/LLM applications using LangChain."""
}

for filepath,content in sample_texts.items():
    with open(filepath,'w',encoding="utf-8") as f:
        f.write(content)

print("File content updated successfully")

File content updated successfully


### TextLoader-Read Single File

In [6]:
from langchain_community.document_loaders import TextLoader
loader=TextLoader("data/text_files/python_intro.txt",encoding="utf-8")
documents=loader.load()
print(type(documents))
print(documents)

print(f"Loaded {len(documents)} document")
print(f"Content Preview:{documents[0].page_content[:100]}")
print(f"Metadata:{documents[0].metadata}")

<class 'list'>
[Document(metadata={'source': 'data/text_files/python_intro.txt'}, page_content='For ssomeone with your background in Java, Spring Boot, and Genesys Cloud, Python is particularly useful for automation scripts, Terraform code generation, AWS integrations, API development, and AI/LLM applications using LangChain.')]
Loaded 1 document
Content Preview:For ssomeone with your background in Java, Spring Boot, and Genesys Cloud, Python is particularly us
Metadata:{'source': 'data/text_files/python_intro.txt'}


C:\Users\HOME\AppData\Local\Temp\ipykernel_12604\2515414142.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


### DirectoryLoader - Multiple Text File

In [7]:
from langchain_community.document_loaders import DirectoryLoader

## Load all the text files from the dirctory
directory_loader = DirectoryLoader(
    "data/text_files",
    glob="**/*.txt", ## Pattern to match files
    loader_cls= TextLoader, ## Loader class to use
    loader_kwargs= {'encoding':'utf-8'},
    show_progress=True 
)

documents = directory_loader.load()

print(f"Loaded: {len(documents)}  documents")
for i,doc in enumerate(documents):
    print(f"\n Documents {i+1}:")
    print(f"Source:{doc.metadata['source']}")
    print(f"Length: {len(doc.page_content)} Characters")

100%|██████████| 2/2 [00:00<00:00, 850.77it/s]

Loaded: 2  documents

 Documents 1:
Source:data\text_files\python_intro.txt
Length: 231 Characters

 Documents 2:
Source:data\text_files\rag_intro.txt
Length: 328 Characters


### Text Spliting Strategies

In [8]:
### Different text splitters techniques

from langchain_text_splitters import (
    RecursiveCharacterTextSplitter,
    CharacterTextSplitter,
    TokenTextSplitter
)

print(documents)

[Document(metadata={'source': 'data\\text_files\\python_intro.txt'}, page_content='For ssomeone with your background in Java, Spring Boot, and Genesys Cloud, Python is particularly useful for automation scripts, Terraform code generation, AWS integrations, API development, and AI/LLM applications using LangChain.'), Document(metadata={'source': 'data\\text_files\\rag_intro.txt'}, page_content='Retrieval-Augmented Generation (RAG) is an AI framework that connects Large Language Models (LLMs) to external knowledge bases. Instead of relying solely on pre-trained data, it retrieves relevant facts on demand, allowing the AI to generate highly accurate, up-to-date, and context-aware responses while reducing hallucinations')]


### Character Text Splitter

In [9]:
# Method 1: CharacterTextSplitter
text=documents[0].page_content
print(f"Original Text Length: {len(text)} Characters")
char_splitter = CharacterTextSplitter(
    separator="\n", ## Split on new lines
    chunk_size=50, ## Max characters per chunk
    chunk_overlap=20, ## Overlap between chunks
    length_function=len ## Function to calculate length (default is len for characters
)
char_chunks = char_splitter.split_text(text)
print(f"Number of Chunks: {len(char_chunks)}")
for i,chunk in enumerate(char_chunks):
    print(f"\nChunk {i+1}:")
    print(f"Chunk Length: {len(chunk)} Characters")
    print(f"Chunk Content:\n{chunk}")

Original Text Length: 231 Characters
Number of Chunks: 1

Chunk 1:
Chunk Length: 231 Characters
Chunk Content:
For ssomeone with your background in Java, Spring Boot, and Genesys Cloud, Python is particularly useful for automation scripts, Terraform code generation, AWS integrations, API development, and AI/LLM applications using LangChain.


### Recurrsive Test Splitter

In [10]:
# Method 1: RecursiveCharacterTextSplitter
text=documents[0].page_content
print(f"Original Text Length: {len(text)} Characters")
recursive_splitter = RecursiveCharacterTextSplitter(
    separators=[" "], ## Try splitting on double newlines, then single newlines, then spaces, and finally characters
    chunk_size=80, ## Max characters per chunk
    chunk_overlap=20, ## Overlap between chunks
    length_function=len ## Function to calculate length (default is len for characters)
)
recursive_chunks = recursive_splitter.split_text(text)
print(f"Number of Chunks: {len(recursive_chunks)}")
for i,chunk in enumerate(recursive_chunks):
    print(f"\nChunk {i+1}:")
    print(f"Chunk Length: {len(chunk)} Characters")
    print(f"Chunk Content:\n{chunk}")

Original Text Length: 231 Characters
Number of Chunks: 4

Chunk 1:
Chunk Length: 74 Characters
Chunk Content:
For ssomeone with your background in Java, Spring Boot, and Genesys Cloud,

Chunk 2:
Chunk Length: 72 Characters
Chunk Content:
and Genesys Cloud, Python is particularly useful for automation scripts,

Chunk 3:
Chunk Length: 68 Characters
Chunk Content:
automation scripts, Terraform code generation, AWS integrations, API

Chunk 4:
Chunk Length: 71 Characters
Chunk Content:
integrations, API development, and AI/LLM applications using LangChain.
